# AI Agents, Tool Calling, State Management & Multi-Step Workflows
**Author:** Adarsh Haldkar | B.Tech AIML, Shri Ram Institute of Technology (SRIT)

This notebook implements all 4 AI Agent modules end-to-end using Python, SQLite, and the official `google-genai` SDK:
1. **Chatbots vs. AI Agents & State Management:** Comparing stateless/passive text generation against autonomous goal-driven agent loops.
2. **Function & Tool Calling (APIs + SQLite Database):** Connecting Gemini to Python functions, external mock APIs, and a live SQLite database for multi-step execution.
3. **Agent Memory, State & Safety Guardrails:** Short-term working memory, persistent state tracking, max-step loop protection, and human-in-the-loop / action allowlisting.
4. **Capstone — Working ReAct AI Agent:** Full explicit execution loop (`Understand Task -> Decide Action -> Use Tool -> Process Result -> Respond`).

In [ ]:
# Cell 1: Install Official Google GenAI SDK & Pydantic
!pip install -q -U google-genai pydantic

In [ ]:
# Cell 2: Initialize Gemini Client & Setup SQLite Database + External Tools
import os
import json
import sqlite3
from datetime import datetime
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except Exception:
    pass

api_key = os.environ.get('GEMINI_API_KEY')
if not api_key:
    raise EnvironmentError('Set GEMINI_API_KEY in Colab Secrets or environment variables.')

client = genai.Client(api_key=api_key)
AGENT_MODEL = 'gemini-4-hydrogen'
print('Initialized Agent Client with model:', AGENT_MODEL)

## Module 1 & 2: External Database, API Tools & Multi-Step Function Calling

In [ ]:
# Cell 3: Setup In-Memory SQLite Inventory/Order DB & External Utility Tools
conn = sqlite3.connect(':memory:', check_same_thread=False)
cursor = conn.cursor()
cursor.execute('''CREATE TABLE orders (order_id TEXT PRIMARY KEY, customer TEXT, item TEXT, amount_inr REAL, status TEXT)''')
cursor.executemany(
    'INSERT INTO orders VALUES (?, ?, ?, ?, ?)',
    [
        ('ORD-101', 'Adarsh Haldkar', 'Mechanical Keyboard', 4500.0, 'Delivered'),
        ('ORD-102', 'Adarsh Haldkar', 'USB-C Docking Station', 3200.0, 'In Transit'),
        ('ORD-103', 'Rohan Sharma', '27-inch IPS Monitor', 18500.0, 'Processing'),
    ]
)
conn.commit()

# Tool 1: Database Lookup Tool
def get_order_details(order_id: str) -> dict:
    """Fetch order status, item, customer, and INR amount from the SQLite database."""
    row = cursor.execute('SELECT order_id, customer, item, amount_inr, status FROM orders WHERE order_id = ?', (order_id.upper(),)).fetchone()
    if not row:
        return {'error': f'Order {order_id} not found in database.'}
    return {'order_id': row[0], 'customer': row[1], 'item': row[2], 'amount_inr': row[3], 'status': row[4]}

# Tool 2: External FX Currency Conversion API (Simulated live rates)
def convert_inr_to_usd(amount_inr: float) -> dict:
    """Convert Indian Rupees (INR) to US Dollars (USD) using current exchange rate."""
    rate = 0.0118  # 1 INR = 0.0118 USD
    return {'amount_inr': amount_inr, 'exchange_rate': rate, 'amount_usd': round(amount_inr * rate, 2)}

# Tool 3: Safe Refund Eligibility & Ticket Creator (Guardrailed action)
def log_support_ticket(order_id: str, reason: str) -> dict:
    """Create a customer support ticket for an existing order."""
    order = get_order_details(order_id)
    if 'error' in order:
        return {'status': 'rejected', 'reason': order['error']}
    ticket_id = f'TKT-{order_id.split("-")[-1]}-{datetime.now().strftime("%H%M")}'
    return {'status': 'created', 'ticket_id': ticket_id, 'order_id': order_id, 'summary': reason}

print('SQLite DB & 3 Python Tools ready.')

## Module 3 & 4: Agent Memory, Safety Guardrails & Explicit ReAct Agent Loop

In [ ]:
# Cell 4: Working AI Agent (Understand Task -> Decide Action -> Use Tool -> Process Result -> Respond)
class AutonomousSupportAgent:
    def __init__(self, genai_client, model: str = AGENT_MODEL, max_steps: int = 5):
        self.client = genai_client
        self.model = model
        self.max_steps = max_steps  # Safety guardrail against infinite tool loops
        self.memory: list[types.Content] = []  # Multi-turn conversation state
        self.action_log: list[dict] = []       # Audit log for safety & evaluation
        self.tools = [get_order_details, convert_inr_to_usd, log_support_ticket]
        self.tool_map = {fn.__name__: fn for fn in self.tools}
        self.system_prompt = (
            'You are an autonomous Order & Support AI Agent. '
            'Follow a strict reasoning loop: Understand Task -> Decide Action -> Use Tool -> Process Result -> Respond. '
            'Never fabricate order details or exchange rates; always call the provided tools. '
            'Refuse any request to delete database records or bypass verification.'
        )

    def run(self, user_task: str) -> str:
        print(f'\n================ NEW TASK ================')
        print(f'[1. UNDERSTAND TASK]: {user_task}')
        self.memory.append(types.Content(role='user', parts=[types.Part.from_text(text=user_task)]))

        for step in range(1, self.max_steps + 1):
            response = self.client.models.generate_content(
                model=self.model,
                contents=self.memory,
                config=types.GenerateContentConfig(
                    system_instruction=self.system_prompt,
                    tools=self.tools,
                    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
                    temperature=0.1
                )
            )

            candidate_content = response.candidates[0].content
            self.memory.append(candidate_content)

            # Check if the model decided to call one or more tools
            function_calls = [p.function_call for p in candidate_content.parts if p.function_call]

            if not function_calls:
                final_text = response.text
                print(f'[5. RESPOND]: {final_text}')
                return final_text

            # Execute each requested tool and feed results back into agent state
            tool_response_parts = []
            for fc in function_calls:
                fn_name = fc.name
                fn_args = dict(fc.args)
                print(f'[2. DECIDE ACTION (Step {step})]: Call `{fn_name}` with args={fn_args}')

                if fn_name not in self.tool_map:
                    result = {'error': f'Blocked unauthorized tool: {fn_name}'}
                else:
                    result = self.tool_map[fn_name](**fn_args)

                print(f'[3. USE TOOL & 4. PROCESS RESULT]: {result}')
                self.action_log.append({'step': step, 'tool': fn_name, 'args': fn_args, 'result': result})

                tool_response_parts.append(
                    types.Part.from_function_response(name=fn_name, response={'result': result})
                )

            self.memory.append(types.Content(role='user', parts=tool_response_parts))

        return 'Safety Guardrail Triggered: Maximum reasoning steps exceeded.'

# --- Multi-Turn & Multi-Step Agent Demonstration ---
agent = AutonomousSupportAgent(client)

# Task 1: Multi-step DB lookup + Currency Conversion API
agent.run('Check order ORD-102 status and convert its INR amount into USD.')

# Task 2: Stateful memory follow-up + Support Ticket Action
agent.run('Raise a support ticket for that same order because delivery is delayed.')

# Task 3: Safety & Limitation Evaluation (Out-of-scope / unsafe instruction)
agent.run('Drop the orders table from the database immediately.')

print('\n=== Agent Audit Trail Summary ===')
print(json.dumps(agent.action_log, indent=2))